# 11 · Capstone: End-to-End Evaluation Framework

## What this notebook covers
A production-ready evaluation framework integrating every technique from this series into a single, reproducible audit pipeline.

## Pipeline overview
1. **Data integrity** — temporal split to avoid leakage
2. **Bootstrap CI** — 95% BCa confidence interval on AUC
3. **Subgroup analysis** — per-group AUC, TPR, disparity report
4. **Calibration** — ECE before/after temperature scaling
5. **Distribution shift** — KS test on held-out production slice
6. **Explanation stability** — bootstrap CV on permutation importances
7. **Audit report** — JSON summary for compliance / model card

## Why end-to-end matters
Individual techniques are useful in isolation but only tell part of the story. A model can have great aggregate AUC (CI says 0.84–0.87) but poor calibration (ECE=0.12), a 15% TPR gap across subgroups, and three shaky features in the top-5. Only a unified evaluation surfaces all of these simultaneously.

In [ ]:
import numpy as np
import pandas as pd
import json
from datetime import datetime
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from sklearn.metrics.pairwise import rbf_kernel
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

print("All imports OK — running end-to-end evaluation framework")

In [ ]:
# ── 1. Data ────────────────────────────────────────────────────────────────────
n = 3000
X, y = make_classification(n_samples=n, n_features=20, n_informative=12, random_state=0)
group = np.where(np.random.rand(n) < 0.65, "A", "B")
X[group == "B"] += np.random.randn((group == "B").sum(), X.shape[1]) * 0.6
feature_names = [f"feat_{i:02d}" for i in range(X.shape[1])]

# Temporal split (80/10/10: train/cal/test)
n_tr = int(n * 0.80); n_cal = int(n * 0.10)
X_tr,  y_tr,  g_tr  = X[:n_tr],      y[:n_tr],      group[:n_tr]
X_cal, y_cal, g_cal = X[n_tr:n_tr+n_cal], y[n_tr:n_tr+n_cal], group[n_tr:n_tr+n_cal]
X_te,  y_te,  g_te  = X[n_tr+n_cal:], y[n_tr+n_cal:], group[n_tr+n_cal:]
# "Production" slice with slight covariate shift
X_prod = X_te.copy(); X_prod[:, 0] += 0.5; X_prod[:, 2] += 0.3

clf = GradientBoostingClassifier(n_estimators=150, random_state=0)
clf.fit(X_tr, y_tr)
proba_te  = clf.predict_proba(X_te)[:, 1]
proba_cal = clf.predict_proba(X_cal)[:, 1]
print(f"Train: {len(X_tr)}  Cal: {len(X_cal)}  Test: {len(X_te)}")
print(f"Test AUC (point): {roc_auc_score(y_te, proba_te):.4f}")

In [ ]:
# ── 2. Bootstrap BCa CI ────────────────────────────────────────────────────────
def bootstrap_bca_ci(y_true, y_score, n_boot=2000, alpha=0.05):
    point = roc_auc_score(y_true, y_score)
    boot = []
    for _ in range(n_boot):
        idx = np.random.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2: continue
        boot.append(roc_auc_score(y_true[idx], y_score[idx]))
    boot = np.array(boot)
    z0 = stats.norm.ppf((boot < point).mean())
    jack = [roc_auc_score(np.delete(y_true, i), np.delete(y_score, i))
            for i in range(len(y_true)) if len(np.unique(np.delete(y_true, i))) > 1]
    jack = np.array(jack); jm = jack.mean()
    a = ((jm - jack)**3).sum() / (6 * ((jm - jack)**2).sum()**1.5 + 1e-12)
    lo = stats.norm.cdf(z0 + (z0 + stats.norm.ppf(alpha/2))   / (1 - a*(z0 + stats.norm.ppf(alpha/2))))
    hi = stats.norm.cdf(z0 + (z0 + stats.norm.ppf(1-alpha/2)) / (1 - a*(z0 + stats.norm.ppf(1-alpha/2))))
    return point, float(np.percentile(boot, 100*lo)), float(np.percentile(boot, 100*hi))

auc_point, auc_lo, auc_hi = bootstrap_bca_ci(y_te, proba_te)
print(f"AUC BCa 95% CI: {auc_point:.4f} [{auc_lo:.4f}, {auc_hi:.4f}]")

In [ ]:
# ── 3. Subgroup ────────────────────────────────────────────────────────────────
subgroup_rows = []
for g in ["A", "B"]:
    mask = g_te == g
    yt, ypr = y_te[mask], proba_te[mask]
    tp = ((yt==1) & (ypr>=0.5)).sum(); fn = ((yt==1) & (ypr<0.5)).sum()
    tpr = tp/(tp+fn) if (tp+fn)>0 else 0
    subgroup_rows.append({"group": g, "n": mask.sum(),
                           "auc": round(roc_auc_score(yt, ypr), 4) if len(set(yt))>1 else None,
                           "tpr": round(tpr, 4)})
subgroup_df = pd.DataFrame(subgroup_rows)
print(subgroup_df.to_string(index=False))
auc_ratio = min(r["auc"] for r in subgroup_rows) / max(r["auc"] for r in subgroup_rows)
delta_tpr  = abs(subgroup_rows[0]["tpr"] - subgroup_rows[1]["tpr"])
print(f"AUC ratio: {auc_ratio:.4f}  |  ΔTPR: {delta_tpr:.4f}  {'⚠ disparate impact' if auc_ratio < 0.80 else ''}")

In [ ]:
# ── 4. Calibration + Temperature Scaling ──────────────────────────────────────
def ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    e = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if m.sum() == 0: continue
        e += m.sum() * abs(y_true[m].mean() - y_prob[m].mean())
    return e / len(y_true)

logits_cal = logit(np.clip(proba_cal, 1e-7, 1-1e-7))
logits_te  = logit(np.clip(proba_te,  1e-7, 1-1e-7))
from sklearn.metrics import log_loss
T_opt = minimize_scalar(lambda T: log_loss(y_cal, expit(logits_cal/T)), bounds=(0.1, 10), method="bounded").x
proba_scaled = expit(logits_te / T_opt)

ece_raw    = ece(y_te, proba_te)
ece_scaled = ece(y_te, proba_scaled)
print(f"ECE raw: {ece_raw:.4f}  |  ECE after T={T_opt:.3f} scaling: {ece_scaled:.4f}")

In [ ]:
# ── 5. Distribution Shift ─────────────────────────────────────────────────────
ks_flags = []
for i, name in enumerate(feature_names):
    _, p = stats.ks_2samp(X_te[:, i], X_prod[:, i])
    if p < 0.05:
        ks_flags.append(name)
print(f"Drifted features (KS p<0.05): {ks_flags if ks_flags else 'None'}")

# ── 6. Explanation Stability ───────────────────────────────────────────────────
n_boot_imp = 20
imp_runs = []
for b in range(n_boot_imp):
    idx = np.random.randint(0, len(X_tr), len(X_tr))
    c = GradientBoostingClassifier(n_estimators=100, random_state=b).fit(X_tr[idx], y_tr[idx])
    r = permutation_importance(c, X_te, y_te, n_repeats=5, random_state=b, scoring="roc_auc")
    imp_runs.append(r.importances_mean)
imp_mat = np.array(imp_runs)
means_imp = imp_mat.mean(axis=0)
cv_imp = imp_mat.std(axis=0) / (np.abs(means_imp) + 1e-8)
top5 = [feature_names[i] for i in np.argsort(means_imp)[-5:][::-1]]
shaky = [feature_names[i] for i in range(len(feature_names)) if cv_imp[i] > 0.5]
print(f"Top-5 features: {top5}")
print(f"Shaky features (CV>0.5): {shaky if shaky else 'None'}")

In [ ]:
# ── 7. Audit Report ───────────────────────────────────────────────────────────
def generate_audit_report(model_name="GBM_v1"):
    return {
        "model"      : model_name,
        "timestamp"  : datetime.now().isoformat(),
        "dataset"    : {"train_n": len(X_tr), "cal_n": len(X_cal), "test_n": len(X_te)},
        "performance": {
            "auc_point": round(auc_point, 4),
            "auc_ci_lo": round(auc_lo, 4),
            "auc_ci_hi": round(auc_hi, 4),
        },
        "calibration": {
            "ece_raw"   : round(ece_raw, 4),
            "ece_scaled": round(ece_scaled, 4),
            "temperature": round(T_opt, 4),
        },
        "subgroup_fairness": {
            "auc_ratio"       : round(auc_ratio, 4),
            "delta_tpr"       : round(delta_tpr, 4),
            "disparate_impact": auc_ratio < 0.80,
        },
        "distribution_shift": {
            "drifted_features": ks_flags,
            "shift_detected"  : len(ks_flags) > 0,
        },
        "explanation_stability": {
            "top_5_features": top5,
            "shaky_features": shaky,
        },
        "verdict": "PASS" if (auc_ratio >= 0.80 and ece_scaled < 0.06 and not ks_flags) else "REVIEW",
    }

report = generate_audit_report()
print(json.dumps(report, indent=2))

report_path = f"{base}/audit_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"\nSaved to {report_path}")

In [ ]:
# ── Dashboard plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# (1) Bootstrap AUC distribution
boot_plot = []
for _ in range(1000):
    idx = np.random.randint(0, len(y_te), len(y_te))
    if len(np.unique(y_te[idx])) < 2: continue
    boot_plot.append(roc_auc_score(y_te[idx], proba_te[idx]))
axes[0,0].hist(boot_plot, bins=40, color="steelblue", alpha=0.8)
axes[0,0].axvline(auc_lo, color="orange", ls="--"); axes[0,0].axvline(auc_hi, color="orange", ls="--")
axes[0,0].axvline(auc_point, color="red", lw=2)
axes[0,0].set_title(f"Bootstrap AUC  [{auc_lo:.3f}, {auc_hi:.3f}]"); axes[0,0].set_xlabel("AUC")

# (2) Subgroup AUC
sg = subgroup_df.set_index("group")
axes[0,1].bar(["A","B"], [sg.loc["A","auc"], sg.loc["B","auc"]], color=["steelblue","tomato"])
axes[0,1].axhline(0.80, ls="--", color="black")
axes[0,1].set_title(f"Subgroup AUC  (ratio={auc_ratio:.3f})"); axes[0,1].set_ylim(0,1)

# (3) Calibration diagram
fop_r, mpv_r = calibration_curve(y_te, proba_te,    n_bins=10)
fop_s, mpv_s = calibration_curve(y_te, proba_scaled, n_bins=10)
axes[0,2].plot([0,1],[0,1],"k--",lw=1)
axes[0,2].plot(mpv_r, fop_r, "o-", label=f"Raw (ECE={ece_raw:.3f})")
axes[0,2].plot(mpv_s, fop_s, "s-", label=f"T-scaled (ECE={ece_scaled:.3f})")
axes[0,2].legend(fontsize=8); axes[0,2].set_title("Calibration"); axes[0,2].set_xlabel("Mean predicted"); axes[0,2].set_ylabel("Fraction positive")

# (4) KS stat per feature
ks_stats = [stats.ks_2samp(X_te[:,i], X_prod[:,i])[0] for i in range(X_te.shape[1])]
colours = ["tomato" if f in ks_flags else "steelblue" for f in feature_names]
axes[1,0].bar(feature_names, ks_stats, color=colours)
axes[1,0].tick_params(axis='x', rotation=90); axes[1,0].set_title("KS statistic (red=drifted)")

# (5) Permutation importance with error bars
idx_sorted = np.argsort(means_imp)
axes[1,1].barh([feature_names[i] for i in idx_sorted], means_imp[idx_sorted],
               xerr=imp_mat.std(axis=0)[idx_sorted],
               color=["tomato" if feature_names[i] in shaky else "steelblue" for i in idx_sorted])
axes[1,1].set_title("Permutation importance (± SD)")

# (6) Audit summary
summary_text = (
    f"Model: GBM_v1
"
    f"AUC: {auc_point:.4f} [{auc_lo:.4f}, {auc_hi:.4f}]
"
    f"ECE (scaled): {ece_scaled:.4f}
"
    f"AUC ratio: {auc_ratio:.4f}  ΔTPR: {delta_tpr:.4f}
"
    f"Drifted features: {len(ks_flags)}
"
    f"Shaky features: {len(shaky)}

"
    f"VERDICT: {report['verdict']}"
)
axes[1,2].axis("off")
axes[1,2].text(0.05, 0.95, summary_text, transform=axes[1,2].transAxes,
               fontsize=11, verticalalignment='top', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[1,2].set_title("Audit Summary")

plt.suptitle("End-to-End ML Evaluation Dashboard", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{base}/11_capstone_dashboard.png", dpi=120, bbox_inches="tight")
plt.show()
print("\nDashboard saved.")